In [50]:
import pandas as pd
from tqdm import tqdm

In [51]:
df_ground_truth = pd.read_csv("data/ground_truth.csv") #-data.csv")

In [52]:
df_ground_truth.shape

(1196, 2)

In [53]:
df_ground_truth.head()

,question,document
0,I’ve got about 25 minutes and some apples at h...,2
1,Can you suggest a quick applesauce recipe that...,2
2,I’ve got about an hour and a bunch of apples a...,5
3,Can you suggest a quick baked apple dessert us...,5
4,Got a few apples and butter at home — what’s a...,9


In [54]:
ground_truth = df_ground_truth.to_dict(orient="records")


In [55]:
from ingest import load_data, build_index

documents = load_data()
index = build_index(documents)

In [56]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [70]:
boost = {'total_time': 1.0, 'ingredients': 3.0}

index.search(
    "Any quick apple dessert or snack recipe with just apples, sugar, cinnamon, and water?",
    num_results=5,
    boost_dict=boost
)

[{'id': '126',
  'recipe_name': 'Apple Cinnamon Oatmeal Muffins',
  'total_time': '35 mins',
  'servings': '12',
  'ingredients': '¼ cup quick-cooking oats, 1 tablespoon brown sugar, 1 tablespoon melted butter, ¼ teaspoon ground cinnamon',
  'directions': 'Preheat the oven to 400 degrees F (200 degrees C). Grease a 12-cup muffin pan or line with paper liners.\nMake topping: Mix oats, brown sugar, melted butter, and cinnamon in a small bowl; set aside.\nMake muffins: Whisk together oats, flour, brown sugar, cinnamon, baking powder, baking soda, and salt in a large bowl. Whisk together applesauce, milk, oil, egg, and vanilla extract in a medium bowl. Stir applesauce mixture into flour mixture until all ingredients are moistened. Fold in chopped apple.\nSpoon batter into the prepared muffin cups about 2/3 full; sprinkle oat topping mixture evenly over each muffin.\nBake in the preheated oven until a toothpick inserted near the center comes out clean, about 15 minutes.',
  'nutrition': 'To

In [84]:
def text_search(query):
    boost_dict = {'total_time': 1.0, 'ingredients': 1.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [88]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    print(doc_id)
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        print(d["id"])
        relevance.append(int(int(d["id"]) == int(doc_id)))

    return relevance

In [89]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [90]:
compute_relevance_total(ground_truth[:10], text_search)

100%|██████████| 10/10 [00:00<00:00, 61.67it/s]

2
2
64
33
95
112
2
127
764
353
326
512
5
5
67
36
98
120
5
426
136
11
73
42
9
1042
493
71
9
102
9
677
621
665
24
117
10
140
417
852
136
1041
10
1058
852
619
1041
17
11
142
86
55
24
117
11
1058
852
1041
126
431


[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [91]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  1%|▏         | 15/1196 [00:00<00:08, 138.92it/s]

2
2
64
33
95
112
2
127
764
353
326
512
5
5
67
36
98
120
5
426
136
11
73
42
9
1042
493
71
9
102
9
677
621
665
24
117
10
140
417
852
136
1041
10
1058
852
619
1041
17
11
142
86
55
24
117
11
1058
852
1041
126
431
14
493
1042
40
9
102
14
126
677
593
624
621
16
58
27
89
120
110
16
14
76
45
107
78
17
17
48
79
110
414
17
1041
852
1067
624
126
19
808
19
50
112
81
19
619
852
1041
19
50
20
20
51
82
113
117
20
677
621
852
593
619
24
55
86
117
24
28
24
126
86
55
24
117
26
26
57
119
88
839
26
88
26
119
57
852
27
27
120
58
89
1041
27
58
120
89
27
17
28
55
86
117
24
779
28


  4%|▎         | 43/1196 [00:00<00:08, 134.97it/s]

808
852
353
631
140
29
142
60
29
122
91
29
852
619
764
529
1058
30
493
1042
123
92
61
30
61
30
92
123
677
33
808
2
64
95
33
33
142
126
29
60
122
36
852
1041
5
36
67
36
1064
307
27
58
120
40
1042
493
142
9
40
40
677
621
1058
593
1064
41
417
140
624
852
142
41
140
621
677
593
147
42
142
24
117
55
86
42
852
163
768
619
683
45
493
1042
142
71
40
45
677
621
796
1041
852
47
27
58
89
120
677
47
27
89
58
120
1052
48
48
79
110
17
27
48
58
120
27
89
48
50
50
81
112
19
808
50
529
136
761
42
73
51
126
55
117
24
86
51
1042
493
1064
764
941
55
24
55
86
117
758
55
431
126
24
55
86
57


  6%|▌         | 70/1196 [00:00<00:09, 124.29it/s]

...

594
1021
1032
380
800
1032
677
621
619
593
1032
1033
1041
1042
762
852
332
1033
621
677
1033
1032
593
1035
353
873
1035
952
362
1035
660
529
1043
653
683
1039
1039
808
619
854
1052
1039
677
621
1058
619
1039
1040
1040
312
334
629
1047
1040
1040
1058
353
213
317
1041
1041
736
721
267
27
1041
1041
852
660
353
1032
1042
1042
493
288
996
1013
1042
1042
1056
762
332
232
1043
1043
878
1027
1035
873
1043
1043
873
821
1035
1058
1044
1042
493
1044
238
764
1044
1044
235
439
1058
852
1045
1045
808
339
444
619
1045
1045
621
677
593
1013
1046
1046
807
307
498
600
1046
1046
260
513
553
498
1047
1047
998
235
839
765
1047
1001
910
335
512
711
1050
1050
873
220
176
353
1050
1050
644
1017
353
1001
1052
1052
808
1039
339
444
1052


 99%|█████████▉| 1185/1196 [00:09<00:00, 123.87it/s]

27
120
89
58
1052
1053
1001
910
335
317
711
1053
1053
683
1039
354
1024
1054
332
1054
529
535
718
1054
1046
817
644
513
553
1056
1056
1075
1070
288
446
1056
1075
1056
1070
353
1058
1058
1058
873
353
821
684
1058
1056
796
1067
1058
271
1060
975
1065
1060
808
1070
1060
677
621
852
619
761
1064
1075
1070
1056
746
326
1064
1056
1075
1070
677
621
1065
1065
808
982
975
1060
1065
1080
1065
1058
1060
984
1067
1067
721
574
1082
665
1067
1067
619
1058
1056
624
1070
1070
808
1075
444
926
1070
1056
1075
1070
621
677
1075
1001
910
335
711
661
1075
1075
1056
1070
1058
1064
1077
808
761
982
444
1070
1077
949
534
1077
353
326
1080


100%|██████████| 1196/1196 [00:10<00:00, 119.52it/s]

1001
910
1080
979
512
1080
996
1056
948
984
796
1082
1082
721
835
737
665
1082
353
521
126
619
949
1084
1084
190
952
873
821
1084
677
190
1084
1017
621
1085
1085
980
969
721
1041
1085
1058
1056
949
1067
619
1087
1087
721
746
930
851
1087
1087
1058
1056
1064
238


In [92]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [93]:
relevance_total[:30]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [94]:
hit_rate(relevance_total)

0.7332775919732442

In [69]:
len(relevance_total)

1196

In [67]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [68]:
mrr(relevance_total)

0.0

In [42]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [43]:
evaluate(ground_truth, text_search)

100%|██████████| 1196/1196 [00:07<00:00, 153.37it/s]


{'hit_rate': 0.0, 'mrr': 0.0}

In [45]:
df_validation = df_ground_truth[:700]
df_test = df_ground_truth[700:]

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

In [46]:
def search_boosts(query, recipe_name_boost, ingredients_boost, nutrition_boost):
    boost_dict = {
        "recipe_name": recipe_name_boost,
        "ingredients": ingredients_boost,
        "nutrition": nutrition_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [47]:
results = []

for recipe_name_boost in [0.5, 1.0, 2.0, 3.0]:
    for ingredients_boost in [0.5, 1.0, 2.0, 5.0, 10.0]:
        for nutrition_boost in [0.5, 1.0, 3.0]:
            print(f"Evaluating recipe_name_boost={recipe_name_boost}, ingredients_boost={ingredients_boost}, nutrition_boost={nutrition_boost}...")
            result = evaluate(
                gt_val,
                lambda query, recipe_name_boost=recipe_name_boost, ingredients_boost=ingredients_boost, nutrition_boost=nutrition_boost: search_boosts(
                    query,
                    recipe_name_boost,
                    ingredients_boost,
                    nutrition_boost
                )
            )

            results.append({
                "recipe_name": recipe_name_boost,
                "ingredients": ingredients_boost,
                "nutrition": nutrition_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating recipe_name_boost=0.5, ingredients_boost=0.5, nutrition_boost=0.5...


100%|██████████| 700/700 [00:04<00:00, 144.44it/s]


Evaluating recipe_name_boost=0.5, ingredients_boost=0.5, nutrition_boost=1.0...


100%|██████████| 700/700 [00:04<00:00, 152.53it/s]


Evaluating recipe_name_boost=0.5, ingredients_boost=0.5, nutrition_boost=3.0...


...

100%|██████████| 700/700 [00:04<00:00, 164.95it/s]


Evaluating recipe_name_boost=3.0, ingredients_boost=10.0, nutrition_boost=1.0...


100%|██████████| 700/700 [00:04<00:00, 163.28it/s]


Evaluating recipe_name_boost=3.0, ingredients_boost=10.0, nutrition_boost=3.0...


100%|██████████| 700/700 [00:04<00:00, 159.65it/s]


In [49]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,recipe_name,ingredients,nutrition,hit_rate,mrr
0,0.5,0.5,0.5,0.0,0.0
1,0.5,0.5,1.0,0.0,0.0
2,0.5,0.5,3.0,0.0,0.0
3,0.5,1.0,0.5,0.0,0.0
4,0.5,1.0,1.0,0.0,0.0
5,0.5,1.0,3.0,0.0,0.0
6,0.5,2.0,0.5,0.0,0.0
7,0.5,2.0,1.0,0.0,0.0
8,0.5,2.0,3.0,0.0,0.0
9,0.5,5.0,0.5,0.0,0.0
